# Biblioteca Request
### A ideia central é utilizar a biblioteca Requests para realizar a extração de dados de forma prática e eficiente. 
### Como exemplo, iremos buscar informações diretamente do Databricks, demonstrando como estabelecer a conexão e recuperar os dados em nuvem necessários para análise.

In [5]:
# Importando as bibliotecas necessarias
import pandas as pd
import requests
import time
from dotenv import load_dotenv 
import os


In [ ]:
# Chaves de acesso
load_dotenv()  # Carrega as variáveis de ambiente do arquivo .env

URL =  os.getenv("URL")
TOKEN = os.getenv("TOKEN")
WAREHOUSE_ID = os.getenv("WAREHOUSE_ID")

In [ ]:
# Headers para autenticação
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

#Query para ver esta com a conexão
QUERY = 'SELECT * FROM camada_silver.produtos_silver LIMIT 10'

#Enviando Query
requisicao_post = requests.post(f"{URL}/api/2.0/sql/statements",
    json={"statement": QUERY, "warehouse_id": WAREHOUSE_ID},
    headers=HEADERS
)
print(requisicao_post.status_code)
if requisicao_post.status_code == 200:
    dados = requisicao_post.json()
    statement_id = dados["statement_id"]
    print("Query enviada. ID:", statement_id)
else:
    print("Erro ao enviar a query:", requisicao_post.status_code)

200
Query enviada. ID: 01f1335c-fc46-1729-a4ea-f3a4531b589f


In [ ]:
#Buscando o resultado (GET)
# Utilizamos o while pq nao sabemos o tempo q pode demorar para a query ser processada, 
# entao vamos ficar verificando o status ate que ela seja concluida
while True:
    requisicao_get = requests.get(
        f"{URL}/api/2.0/sql/statements/{statement_id}",
        headers=HEADERS
    )

    resultado = requisicao_get.json()
    status = resultado["status"]["state"]

    print("Status:", status)

    if status == "SUCCEEDED":
        break

    elif status in ["FAILED", "CANCELED"]:
        print("Erro na execução")
        break

    time.sleep(2)

Status: SUCCEEDED

Resultado:
['1', 'Hic', 'Móveis', '861.60']
['2', 'Reiciendis', 'Beleza', '49.23']
['3', 'Dicta', 'Eletrônicos', '1110.96']
['4', 'Dolorem', 'Alimentos', '444.06']
['5', 'Facilis', 'Roupas', '90.87']
['6', 'Laudantium', 'Móveis', '889.21']
['7', 'Impedit', 'Beleza', '90.44']
['8', 'Placeat', 'Eletrônicos', '89.83']
['9', 'Omnis', 'Alimentos', '166.38']
['10', 'Sit', 'Roupas', '1181.58']


In [58]:
#Transformando o resultado em DataFrame

# dados
linhas = resultado["result"]["data_array"]

# nomes das colunas
colunas=[]
for i in resultado["manifest"]["schema"]["columns"]:
    colunas.append(i["name"])

# criando DataFrame
df = pd.DataFrame(linhas, columns=colunas)
df

,id_produto,nome_produto,categoria,preco_unitario
0,1,Hic,Móveis,861.60
1,2,Reiciendis,Beleza,49.23
2,3,Dicta,Eletrônicos,1110.96
3,4,Dolorem,Alimentos,444.06
4,5,Facilis,Roupas,90.87
5,6,Laudantium,Móveis,889.21
6,7,Impedit,Beleza,90.44
7,8,Placeat,Eletrônicos,89.83
8,9,Omnis,Alimentos,166.38
9,10,Sit,Roupas,1181.58
